# ETTh1 Clean Re-run

Fixes both confounds in the original `ci_cd_etth1_raw_with_budget.csv`:

1. **Batch-size asymmetry:** original used CI batch=128, CD batch=32 (4x more
   CD steps per epoch). This notebook uses **batch=32 for both**, giving
   identical steps-per-epoch so the step-count gap cannot explain any MSE difference.

2. **Warmup asymmetry:** original used CI warmup=10, CD warmup=2, causing CD to
   early-stop at epoch 1 before its LR schedule peaked at H=336 and H=720.
   This notebook uses **warmup_epochs=10 for both**, preserving the original CI
   schedule and giving CD a fair warmup.

3. **min_epochs=15:** prevents early stopping from firing before warmup completes.
   The best checkpoint is still used for test evaluation, so this does not inflate
   test MSE; it only prevents degenerate epoch-1 stops.

**Seeds:** {42, 123, 456, 789, 1011} — n=5, consistent with AR(1) grid and LF sweep.

**Output:** `/kaggle/working/results_etth1.csv`
Schema matches `ci_cd_etth1_raw_with_budget.csv` for easy comparison.

**Runtime:** ~6h on T4 (dominated by CD at batch=32, ~252 steps/epoch).

**Design note:** CD still uses its own internal cosine-warmup LR schedule
(as in all other PatchTST experiments). The only change is that warmup duration
and batch size now match CI, removing both directions of the original confound.


In [ ]:
import os
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")

import gc, random, time
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.amp import GradScaler, autocast
from torch.utils.data import DataLoader, TensorDataset

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {DEVICE}")
if DEVICE.type == "cuda":
    print(f"GPU:  {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")


def set_seed(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


def free_cuda() -> None:
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()


In [ ]:
# ── Configuration ─────────────────────────────────────────────────────────────

# ETTh1 dataset parameters
SEQ_LEN:  int = 512
PRED_LENS: list[int] = [96, 192, 336, 720]
SEEDS:     list[int] = [42, 123, 456, 789, 1011]
MODES:     list[str] = ["CI", "CD"]

# Architecture (Table 2, Synthetic + ETTh1 config)
PATCH_SIZE:   int = 16
PATCH_STRIDE: int = 8
D_MODEL:  int   = 64
N_HEADS:  int   = 8
N_LAYERS: int   = 3
DROPOUT:  float = 0.2

# Training — both modes identical to remove confounds
BATCH_SIZE:     int   = 32    # same for CI and CD (CD's constraint)
LR:             float = 1e-4
WEIGHT_DECAY:   float = 1e-4
MAX_EPOCHS:     int   = 100   # same as original ETTh1
PATIENCE:       int   = 10
GRAD_CLIP:      float = 1.0
WARMUP_EPOCHS:  int   = 10    # same for CI and CD (CI's original schedule)
MIN_EPOCHS:     int   = 15    # prevents early stopping before warmup completes

# ETTh1 standard split (timesteps)
TRAIN_END: int = 8640
VAL_END:   int = 8640 + 2880   # 11520

N_PATCHES = (SEQ_LEN - PATCH_SIZE) // PATCH_STRIDE + 1
print(f"N_PATCHES={N_PATCHES}  batch={BATCH_SIZE}  warmup={WARMUP_EPOCHS}  min_epochs={MIN_EPOCHS}")
print(f"Total runs: {len(PRED_LENS)*len(MODES)*len(SEEDS)}")


In [ ]:
# ── Dataset loader ────────────────────────────────────────────────────────────

def load_etth1() -> np.ndarray:
    """Load ETTh1 from Kaggle input or working directory.
    Returns (T, 7) float64 array, all 7 variates."""
    candidates = [
        "/kaggle/input/etth1/ETTh1.csv",
        "/kaggle/input/ett-small/ETTh1.csv",
        "/kaggle/input/etthdata/ETTh1.csv",
        "/kaggle/working/ETTh1.csv",
    ]
    for p in candidates:
        if Path(p).exists():
            df = pd.read_csv(p)
            # Drop date column if present
            num_cols = df.select_dtypes(include=np.number).columns.tolist()
            print(f"Loaded ETTh1 from {p}: {len(df)} rows, {len(num_cols)} variates")
            return df[num_cols].values.astype(np.float64)
    raise FileNotFoundError(
        "ETTh1.csv not found. Upload it as a Kaggle dataset input "
        "or copy to /kaggle/working/ETTh1.csv"
    )


def split_normalise_etth1(
    data: np.ndarray, pred_len: int
) -> tuple[np.ndarray, np.ndarray, np.ndarray]:
    """Standard ETTh1 splits with z-score fit on train only."""
    train = data[:TRAIN_END]
    val   = data[TRAIN_END:VAL_END]
    test  = data[VAL_END:]
    mean  = train.mean(axis=0, keepdims=True)
    std   = train.std(axis=0, keepdims=True)
    std   = np.where(std == 0, 1.0, std)
    return (train-mean)/std, (val-mean)/std, (test-mean)/std


def make_windows(data: np.ndarray, pred_len: int) -> tuple[torch.Tensor, torch.Tensor]:
    """Stride-tricks windowing — no Python loop."""
    T, C = data.shape
    n = T - SEQ_LEN - pred_len + 1
    if n <= 0:
        raise ValueError(f"Not enough data: T={T}, seq={SEQ_LEN}, pred={pred_len}")
    s0, s1 = data.strides
    view = np.lib.stride_tricks.as_strided(
        data, shape=(n, SEQ_LEN + pred_len, C), strides=(s0, s0, s1))
    xs = np.ascontiguousarray(view[:, :SEQ_LEN])
    ys = np.ascontiguousarray(view[:, SEQ_LEN:])
    return torch.tensor(xs, dtype=torch.float32), torch.tensor(ys, dtype=torch.float32)


# Pre-load the dataset once
RAW_DATA = load_etth1()
N_VARIATES = RAW_DATA.shape[1]
print(f"ETTh1: {RAW_DATA.shape}  variates={N_VARIATES}")


In [ ]:
# ── Model definitions ─────────────────────────────────────────────────────────

class PatchEmbedding(nn.Module):
    def __init__(self) -> None:
        super().__init__()
        self.proj    = nn.Linear(PATCH_SIZE, D_MODEL)
        self.dropout = nn.Dropout(DROPOUT)
    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.dropout(self.proj(x))


class PatchTST_CI(nn.Module):
    """Channel-independent PatchTST. (B,L,C) -> (B,pred_len,C)."""
    def __init__(self, pred_len: int) -> None:
        super().__init__()
        self.pred_len = pred_len
        self.embed    = PatchEmbedding()
        layer = nn.TransformerEncoderLayer(
            d_model=D_MODEL, nhead=N_HEADS, dim_feedforward=D_MODEL*4,
            dropout=DROPOUT, batch_first=True)
        self.encoder = nn.TransformerEncoder(layer, num_layers=N_LAYERS)
        self.head    = nn.Linear(N_PATCHES * D_MODEL, pred_len)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        B, L, C = x.shape
        x  = x.permute(0,2,1).reshape(B*C, L)
        p  = x.unfold(-1, PATCH_SIZE, PATCH_STRIDE)
        e  = self.encoder(self.embed(p))
        return self.head(e.reshape(B*C,-1)).reshape(B,C,-1).permute(0,2,1)


class PatchTST_CD(nn.Module):
    """Channel-dependent PatchTST with per-variate head. (B,L,C) -> (B,pred_len,C)."""
    def __init__(self, pred_len: int) -> None:
        super().__init__()
        self.pred_len = pred_len
        self.embed    = PatchEmbedding()
        layer = nn.TransformerEncoderLayer(
            d_model=D_MODEL, nhead=N_HEADS, dim_feedforward=D_MODEL*4,
            dropout=DROPOUT, batch_first=True)
        self.encoder = nn.TransformerEncoder(layer, num_layers=N_LAYERS)
        self.head    = nn.Linear(N_PATCHES * D_MODEL, pred_len)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        B, L, C = x.shape
        xp  = x.permute(0,2,1).reshape(B*C, L)
        p   = xp.unfold(-1, PATCH_SIZE, PATCH_STRIDE)
        emb = self.embed(p).reshape(B, C, N_PATCHES, -1)
        seq = emb.reshape(B, C*N_PATCHES, -1)
        enc = self.encoder(seq).reshape(B*C, -1)
        return self.head(enc).reshape(B,C,-1).permute(0,2,1)


def build_model(mode: str, pred_len: int) -> nn.Module:
    if mode == "CI": return PatchTST_CI(pred_len)
    if mode == "CD": return PatchTST_CD(pred_len)
    raise ValueError(f"Unknown mode: {mode}")


# Architecture assertion: CD head must be per-variate Linear(N*D, pred_len)
for pl in PRED_LENS:
    _cd = build_model("CD", pl)
    assert _cd.head.in_features  == N_PATCHES * D_MODEL, _cd.head
    assert _cd.head.out_features == pl, _cd.head
    del _cd
free_cuda()
print(f"CD head assertions passed for all pred_lens: {PRED_LENS}")


In [ ]:
# ── Training engine ───────────────────────────────────────────────────────────

def _cosine_warmup(opt, epoch: int, warmup: int) -> None:
    if epoch < warmup:
        lr = LR * (epoch + 1) / warmup
    else:
        p  = (epoch - warmup) / max(1, MAX_EPOCHS - warmup)
        lr = LR * 0.5 * (1.0 + np.cos(np.pi * p))
    for g in opt.param_groups:
        g["lr"] = lr


@torch.no_grad()
def _eval(model: nn.Module, loader: DataLoader) -> tuple[float, float]:
    model.eval()
    mse = mae = n = 0.0
    for xb, yb in loader:
        pred = model(xb.to(DEVICE)).cpu()
        mse += nn.functional.mse_loss(pred, yb, reduction="sum").item()
        mae += nn.functional.l1_loss(pred,  yb, reduction="sum").item()
        n   += yb.numel()
    return mse/n, mae/n


def _fit(mode: str, pred_len: int, seed: int,
         datasets: tuple) -> dict:
    x_tr,y_tr,x_va,y_va,x_te,y_te = datasets
    tr_dl = DataLoader(TensorDataset(x_tr,y_tr), batch_size=BATCH_SIZE,
                       shuffle=True,  drop_last=False)
    va_dl = DataLoader(TensorDataset(x_va,y_va), batch_size=BATCH_SIZE,
                       shuffle=False, drop_last=False)
    te_dl = DataLoader(TensorDataset(x_te,y_te), batch_size=BATCH_SIZE,
                       shuffle=False, drop_last=False)

    model  = build_model(mode, pred_len).to(DEVICE)
    opt    = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
    scaler = GradScaler("cuda", enabled=(DEVICE.type=="cuda"))
    crit   = nn.MSELoss()

    spe = len(tr_dl)
    best_val = float("inf"); best_ep = 0; best_steps = 0
    best_state = None; no_imp = 0; total = 0

    try:
        for epoch in range(MAX_EPOCHS):
            _cosine_warmup(opt, epoch, WARMUP_EPOCHS)
            model.train()
            for xb,yb in tr_dl:
                opt.zero_grad(set_to_none=True)
                with autocast("cuda", enabled=(DEVICE.type=="cuda")):
                    loss = crit(model(xb.to(DEVICE)), yb.to(DEVICE))
                scaler.scale(loss).backward()
                scaler.unscale_(opt)
                nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
                scaler.step(opt); scaler.update()
                total += 1

            val_mse, _ = _eval(model, va_dl)
            if val_mse < best_val:
                best_val=val_mse; best_ep=epoch+1; best_steps=total
                best_state={k:v.cpu().clone() for k,v in model.state_dict().items()}
                no_imp=0
            else:
                # Early stopping only after MIN_EPOCHS to prevent premature stops
                if epoch + 1 >= MIN_EPOCHS:
                    no_imp += 1
                    if no_imp >= PATIENCE:
                        break
    finally:
        if best_state:
            model.load_state_dict(best_state)
        test_mse, test_mae = _eval(model, te_dl)
        del model, opt, scaler
        free_cuda()

    return {"mode": mode, "pred_len": pred_len, "seed": seed,
            "test_mse": test_mse, "test_mae": test_mae,
            "best_epoch": best_ep, "batch_size": BATCH_SIZE,
            "steps_per_epoch": spe, "total_steps_to_best": best_steps,
            "warmup_epochs": WARMUP_EPOCHS, "min_epochs": MIN_EPOCHS,
            "max_epochs": MAX_EPOCHS}


def train_one(mode: str, pred_len: int, seed: int) -> dict:
    """Build dataset and fit. OOM-safe (raises; batch is already matched)."""
    set_seed(seed)
    tr, va, te = split_normalise_etth1(RAW_DATA, pred_len)
    datasets = (*make_windows(tr, pred_len),
                *make_windows(va, pred_len),
                *make_windows(te, pred_len))
    set_seed(seed)   # re-seed so model init is identical to any retry
    return _fit(mode, pred_len, seed, datasets)


In [ ]:
# ── Main sweep ────────────────────────────────────────────────────────────────
OUT = Path("/kaggle/working/results_etth1.csv")
TMP = OUT.with_suffix(".csv.tmp")


def _key(mode: str, pred_len: int, seed: int) -> tuple:
    return (str(mode), int(pred_len), int(seed))


def _save(rows: list) -> None:
    pd.DataFrame(rows).to_csv(TMP, index=False)
    os.replace(TMP, OUT)


if OUT.exists() and OUT.stat().st_size > 100:
    _ex   = pd.read_csv(OUT)
    done  = {_key(r['mode'],r.pred_len,r.seed) for r in _ex.itertuples()}
    results = _ex.to_dict("records")
    print(f"Resuming: {len(done)} runs done.")
else:
    done, results = set(), []

total = len(PRED_LENS)*len(MODES)*len(SEEDS)
idx, fails = 0, []

for pred_len in PRED_LENS:
    for mode in MODES:
        for seed in SEEDS:
            idx += 1
            key = _key(mode, pred_len, seed)
            if key in done:
                print(f"[{idx}/{total}] SKIP H={pred_len} mode={mode} seed={seed}")
                continue
            print(f"[{idx}/{total}] H={pred_len} mode={mode} seed={seed} ...",
                  end=" ", flush=True)
            t0 = time.time()
            try:
                row = train_one(mode, pred_len, seed)
            except Exception as exc:
                free_cuda()
                print(f"FAILED: {type(exc).__name__}: {exc}")
                fails.append((pred_len, mode, seed, repr(exc)))
                continue
            print(f"mse={row['test_mse']:.4f}  epoch={row['best_epoch']}"
                  f"  steps={row['total_steps_to_best']}  ({time.time()-t0:.0f}s)")
            results.append(row)
            done.add(key)
            _save(results)

print(f"\nDone. {len(results)}/{total} runs -> {OUT}")
if fails:
    print(f"{len(fails)} failures:")
    for f in fails: print(" ", f)


In [ ]:
# ── Summary and comparison with original confounded results ───────────────────
df = pd.read_csv(OUT)
print(f"Rows: {len(df)}  seeds: {sorted(df['seed'].unique())}  modes: {sorted(df['mode'].unique())}")
print(f"warmup_epochs: {df['warmup_epochs'].unique()}  "
      f"batch_size: {df['batch_size'].unique()}  "
      f"min_epochs: {df['min_epochs'].unique()}")

print("\n=== Clean ETTh1: mean test MSE over seeds ===")
piv = df.groupby(["pred_len","mode"])["test_mse"].agg(["mean","std"]).unstack("mode")
print(piv.round(4).to_string())

print("\n=== CD/CI ratio per horizon ===")
for H in PRED_LENS:
    sub = df[df.pred_len==H]
    ci  = sub[sub['mode']=='CI']['test_mse'].mean()
    cd  = sub[sub['mode']=='CD']['test_mse'].mean()
    print(f"  H={H}: CI={ci:.4f}  CD={cd:.4f}  ratio={cd/ci:.4f}")

# Paired t-test per horizon
from scipy import stats
print("\n=== Per-horizon paired CI [{lo:.4f}, {hi:.4f}] (n=5, df=4) ===")
for H in PRED_LENS:
    sub = df[df.pred_len==H]
    ci_s = sub[sub['mode']=='CI'].set_index('seed')['test_mse']
    cd_s = sub[sub['mode']=='CD'].set_index('seed')['test_mse']
    d = (cd_s - ci_s).dropna()
    m, se = d.mean(), d.std(ddof=1)/np.sqrt(len(d))
    t_crit = stats.t.ppf(0.975, len(d)-1)
    lo, hi = m - t_crit*se, m + t_crit*se
    sig = "SIG" if lo > 0 or hi < 0 else "ns"
    print(f"  H={H}: diff={m:+.4f}  95%CI=[{lo:.4f},{hi:.4f}]  {sig}")

# Compare to original confounded results if available
for cand in ["/kaggle/input/previous_results/ci_cd_etth1_raw_with_budget.csv",
             "/kaggle/working/ci_cd_etth1_raw_with_budget.csv"]:
    if Path(cand).exists():
        orig = pd.read_csv(cand)
        print("\n=== Comparison: clean (n=5, matched budget) vs original (n=3, confounded) ===")
        print(f"{'H':>4} {'CI_orig':>9} {'CD_orig':>9} {'CI_clean':>9} {'CD_clean':>9}")
        for H in PRED_LENS:
            ci_o = orig[(orig.pred_len==H)&(orig['mode']=='CI')]['test_mse'].mean()
            cd_o = orig[(orig.pred_len==H)&(orig['mode']=='CD')]['test_mse'].mean()
            ci_c = df[(df.pred_len==H)&(df['mode']=='CI')]['test_mse'].mean()
            cd_c = df[(df.pred_len==H)&(df['mode']=='CD')]['test_mse'].mean()
            print(f"{H:>4} {ci_o:>9.4f} {cd_o:>9.4f} {ci_c:>9.4f} {cd_c:>9.4f}")
        break

print("\n=== Steps per epoch (confirming budget match) ===")
print(df.groupby('mode')['steps_per_epoch'].unique().to_dict())
print("CI and CD should have identical steps_per_epoch.")


In [ ]:
from IPython.display import FileLink
FileLink(str(OUT))
